In [27]:
import json
from pathlib import Path

import joblib
import pandas as pd


In [28]:
# Same names used in training
TARGET = "addicted_label"
ID_COL = "id"

MODEL_PATH = "data/artifacts/addiction_pipeline.joblib"
FEATURES_PATH = "data/artifacts/feature_columns.json"


In [29]:
pipeline = joblib.load(MODEL_PATH)

In [30]:
test_df = pd.read_csv("data/test.csv")

test_X = test_df.drop(
    columns=[ID_COL, TARGET],
    errors="ignore"
)

test_ids = test_df[ID_COL]


In [31]:
# Load feature names saved during training
with open(FEATURES_PATH, "r") as file:
    training_features = json.load(file)


In [32]:
# Check whether test.csv has the same columns as train.csv
missing_columns = set(training_features) - set(test_X.columns)
extra_columns = set(test_X.columns) - set(training_features)

print("\nMissing columns in test.csv:", missing_columns)
print("Extra columns in test.csv:", extra_columns)

if missing_columns:
    raise ValueError(
        "test.csv is missing feature columns required by the trained model."
    )



Missing columns in test.csv: set()
Extra columns in test.csv: set()


In [33]:
# Ensure exact same feature order as training
test_X = test_X.reindex(columns=training_features)

In [34]:
# Predict probability of addicted_label = 1
test_probabilities = pipeline.predict_proba(test_X)[:, 1]

In [35]:
# Optional: hard class predictions, 0 or 1
test_predictions = pipeline.predict(test_X)

In [36]:
# Show a table before saving
prediction_table = pd.DataFrame({
    "id": test_ids,
    "predicted_label_0_or_1": test_predictions,
    "probability_addicted_label_1": test_probabilities
})
print("\nPrediction preview:")
print(prediction_table.head(10))

print("\nProbability summary:")
print(prediction_table["probability_addicted_label_1"].describe())



Prediction preview:
       id  predicted_label_0_or_1  probability_addicted_label_1
0  691369                       1                      0.973292
1  691370                       1                      0.716453
2  691371                       1                      0.968443
3  691372                       1                      0.942292
4  691373                       1                      0.992964
5  691374                       1                      0.678264
6  691375                       1                      0.871834
7  691376                       0                      0.378308
8  691377                       1                      0.950131
9  691378                       1                      0.628869

Probability summary:
count    296302.000000
mean          0.656192
std           0.354865
min           0.000000
25%           0.323364
50%           0.810386
75%           0.988679
max           1.000000
Name: probability_addicted_label_1, dtype: float64


In [37]:
submission = pd.DataFrame({
    "id": test_ids,
    "addicted_label": test_probabilities
})

In [38]:
Path("outputs").mkdir(exist_ok=True)

In [42]:
output_path = Path("data/outputs/submission.csv")

# Create data/outputs/ if it does not exist
output_path.parent.mkdir(parents=True, exist_ok=True)

submission.to_csv(
    output_path,
    index=False
)

print("\nSubmission created: outputs/submission.csv")
print(submission.head(10))



Submission created: outputs/submission.csv
       id  addicted_label
0  691369        0.973292
1  691370        0.716453
2  691371        0.968443
3  691372        0.942292
4  691373        0.992964
5  691374        0.678264
6  691375        0.871834
7  691376        0.378308
8  691377        0.950131
9  691378        0.628869
